# LA Metro Rail / SCVTA are double-counted on Jul 12

* charts jump up quite suddenly, why?
* grab `fct_scheduled_trips`, compare to `fct_schedule_rt_route_direction_summary`
* dig into the quartet, look at the `feed_key-vp_gtfs_key-tu_gtfs_key` combination.
   * why is `feed_key` getting linked to multiple keys, it should be one per day. 

In [2]:
import gcsfs
import pandas as pd

import world_cup_vars as wc_vars
GCS_FILE_PATH = wc_vars.GCS_FILE_PATH

In [31]:
trips_df = pd.read_parquet(
    f"{GCS_FILE_PATH}fct_scheduled_trips_2026-07-12.parquet",
    filesystem = gcsfs.GCSFileSystem()
)

In [32]:
#803 and 807
trips_df.groupby(["name", "route_id"]).agg({
    "trip_id": "nunique",
    "trip_instance_key": "nunique",
}).reset_index()

,name,route_id,trip_id,trip_instance_key
0,LA Metro Rail Schedule,801,220,220
1,LA Metro Rail Schedule,802,121,121
2,LA Metro Rail Schedule,803,200,200
3,LA Metro Rail Schedule,804,172,172
4,LA Metro Rail Schedule,805,194,194
5,LA Metro Rail Schedule,807,198,198
6,SCVTA Schedule,200,3,3
7,SCVTA Schedule,201,1,1
8,SCVTA Schedule,202,1,1
9,SCVTA Schedule,203,2,2


In [50]:
debug_df = pd.read_parquet(
    f"{GCS_FILE_PATH}debug_dim_gtfs_datasets.parquet",
    filesystem = gcsfs.GCSFileSystem()
)

In [28]:
route_df = pd.read_parquet(
    f"{GCS_FILE_PATH}fct_daily_schedule_rt_route_direction_summary_world_cup.parquet",
    filesystem=gcsfs.GCSFileSystem(),
    filters = [[
        ("service_date", "==", pd.to_datetime("2026-07-12").date()),
        ("schedule_name", "in", ["LA Metro Rail Schedule", "SCVTA Schedule"])
    ]]
)

In [63]:
route_df[
    ["schedule_name", "feed_key",
         "vp_gtfs_dataset_key",
         "tu_gtfs_dataset_key", 
         ]].drop_duplicates()

,schedule_name,feed_key,vp_gtfs_dataset_key,tu_gtfs_dataset_key
0,SCVTA Schedule,d062b23c6635fcf9b07204533e1b8e83,77afa24b52e6677d0a6a44f688586d45,d1dad3d5841d7b665f8d99a3c5029977
1,SCVTA Schedule,d062b23c6635fcf9b07204533e1b8e83,None,e14e1800651c69c0f39b0f246027cfce
5,SCVTA Schedule,d062b23c6635fcf9b07204533e1b8e83,4eb71b65c547a8411cdb913c9975b206,e14e1800651c69c0f39b0f246027cfce
6,SCVTA Schedule,d062b23c6635fcf9b07204533e1b8e83,77afa24b52e6677d0a6a44f688586d45,None
114,LA Metro Rail Schedule,0b2690d82204cbfcb0558ca5b156d1cc,e9a66e21e053ddb99df8e00b729a04cb,cf57d72c32a02c05bd6819b0a67ebd09
115,LA Metro Rail Schedule,0b2690d82204cbfcb0558ca5b156d1cc,07c6c0760475ad0633d6a70b3e059fea,ab214c0ad404a7b913d2550ee1892ac7


In [62]:
route_df[route_df.schedule_name == "LA Metro Rail Schedule"][
    ["schedule_name", "feed_key",
         "vp_gtfs_dataset_key",
         "tu_gtfs_dataset_key", 
         ]].drop_duplicates()

,schedule_name,feed_key,vp_gtfs_dataset_key,tu_gtfs_dataset_key
114,LA Metro Rail Schedule,0b2690d82204cbfcb0558ca5b156d1cc,e9a66e21e053ddb99df8e00b729a04cb,cf57d72c32a02c05bd6819b0a67ebd09
115,LA Metro Rail Schedule,0b2690d82204cbfcb0558ca5b156d1cc,07c6c0760475ad0633d6a70b3e059fea,ab214c0ad404a7b913d2550ee1892ac7


In [54]:
list_of_gtfs_keys = [
    i for i in route_df.schedule_gtfs_dataset_key.unique().tolist() + 
    route_df.vp_gtfs_dataset_key.unique().tolist() + 
    route_df.tu_gtfs_dataset_key.unique().tolist() if i is not None
]

In [57]:
debug_df[(debug_df.gtfs_dataset_key.isin(list_of_gtfs_keys)) & (debug_df.name.str.contains("LA Metro"))]

,gtfs_dataset_key,name,type,_valid_from,_valid_to
28,cf57d72c32a02c05bd6819b0a67ebd09,LA Metro Rail Trip Updates,trip_updates,2026-07-13 00:00:00+00:00,2098-12-31 23:59:59.999999+00:00
30,ab214c0ad404a7b913d2550ee1892ac7,LA Metro Rail Trip Updates,trip_updates,2026-05-27 00:00:00+00:00,2026-07-12 23:59:59.999999+00:00
33,e9a66e21e053ddb99df8e00b729a04cb,LA Metro Rail Vehicle Positions,vehicle_positions,2026-07-13 00:00:00+00:00,2098-12-31 23:59:59.999999+00:00
34,07c6c0760475ad0633d6a70b3e059fea,LA Metro Rail Vehicle Positions,vehicle_positions,2026-05-27 00:00:00+00:00,2026-07-12 23:59:59.999999+00:00
51,7ada6c55e4a29f4535e84c504a994b14,LA Metro Rail Schedule,schedule,2025-08-26 00:00:00+00:00,2098-12-31 23:59:59.999999+00:00


In [39]:
route_df[["schedule_name", "feed_key",
         "vp_gtfs_dataset_key", 
         "tu_gtfs_dataset_key", 
         ]].drop_duplicates()

,schedule_name,feed_key,vp_gtfs_dataset_key,tu_gtfs_dataset_key
0,SCVTA Schedule,d062b23c6635fcf9b07204533e1b8e83,77afa24b52e6677d0a6a44f688586d45,d1dad3d5841d7b665f8d99a3c5029977
1,SCVTA Schedule,d062b23c6635fcf9b07204533e1b8e83,None,e14e1800651c69c0f39b0f246027cfce
5,SCVTA Schedule,d062b23c6635fcf9b07204533e1b8e83,4eb71b65c547a8411cdb913c9975b206,e14e1800651c69c0f39b0f246027cfce
6,SCVTA Schedule,d062b23c6635fcf9b07204533e1b8e83,77afa24b52e6677d0a6a44f688586d45,None
114,LA Metro Rail Schedule,0b2690d82204cbfcb0558ca5b156d1cc,e9a66e21e053ddb99df8e00b729a04cb,cf57d72c32a02c05bd6819b0a67ebd09
115,LA Metro Rail Schedule,0b2690d82204cbfcb0558ca5b156d1cc,07c6c0760475ad0633d6a70b3e059fea,ab214c0ad404a7b913d2550ee1892ac7
